# Regression with a Feed-Forward Network

In this notebook you stack layers of neurons into a **feed-forward network**, also called a multilayer perceptron (MLP), and use it to predict house prices with the Keras recipe from notebook 02. A linear regression model is the baseline the network has to outperform.

## Learning Objectives

By the end of this notebook, you should be able to:

- Build, compile and train a feed-forward network for regression with the Keras Sequential API on the California housing data.
- Standardise the input features inside the model with a Keras `Normalization` layer.
- Compare the test RMSE of the network with a linear regression baseline.

## Setup

In [ ]:
import keras
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

keras.utils.set_random_seed(42)

## 1. The California Housing Data

[`fetch_california_housing`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) downloads census data for districts in California (it is cached after the first run). Each row is one district with eight numeric features, and the target is the median house value in units of 100,000 US dollars.

Split the data three ways with [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), applied twice: a **training** set to fit the weights, a **validation** set to watch for overfitting during training, and a **test** set for the final comparison.

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42
)

print("Train:", X_train.shape, " Validation:", X_valid.shape, " Test:", X_test.shape)
X_train.describe().T[["mean", "std", "min", "max"]].round(2)

The features live on very different scales (compare `Population` with `AveBedrms`). Gradient descent works best when every feature has a similar scale, so every model below standardises the inputs first.

## 2. Baseline: Linear Regression

A baseline tells us whether the neural network is worth its extra complexity. We use RMSE (root mean squared error), which is in the same unit as the target.

In [ ]:
lin_reg = make_pipeline(StandardScaler(), LinearRegression())
lin_reg.fit(X_train, y_train)

test_rmse = {"Linear regression": root_mean_squared_error(y_test, lin_reg.predict(X_test))}
print(f"Linear regression test RMSE: {test_rmse['Linear regression']:.3f}")

## 3. A Feed-Forward Network with Keras

The network is built with the Keras recipe from notebook 02, with one new step: the features are standardised inside the model by a [`Normalization`](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/) layer. Its `.adapt()` method learns the mean and variance of every feature from the training set, like the `StandardScaler` in the linear regression pipeline above. The layer then applies these values to every input, so the scaling is saved together with the model.

The network has eight inputs, three hidden layers of 50 ReLU neurons and one output neuron. For regression the output neuron has no activation, so it can return any number:

<img src="../assets/feedforward_regression.png" width="900">

### Build the Model

[`keras.Sequential`](https://keras.io/api/models/sequential/) stacks layers in order. `keras.Input` declares the shape of one example: eight features.

In [ ]:
norm_layer = keras.layers.Normalization()
norm_layer.adapt(X_train.to_numpy())  # learn the mean and variance of each feature

model = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        norm_layer,
        keras.layers.Dense(units=50, activation="relu"),
        keras.layers.Dense(units=50, activation="relu"),
        keras.layers.Dense(units=50, activation="relu"),
        keras.layers.Dense(units=1),
    ]
)
model.summary()

Each `Dense` layer has one weight per input per neuron, plus one bias per neuron. The first hidden layer has $8 \times 50 + 50 = 450$ parameters and the next two have $50 \times 50 + 50 = 2{,}550$ each. The normalisation layer stores a mean and variance for each feature; these are counted as non-trainable because gradient descent does not change them.

### Compile and Train

`compile()` fixes how the model learns: the **loss** to minimise (MSE, as in the previous notebook), the **optimiser** that updates the weights ([Adam](https://keras.io/api/optimizers/adam/), a gradient descent variant that adapts the step size per weight), and the **metrics** to report. One **epoch** is one pass over the whole training set.

In [ ]:
model.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)
history = model.fit(
    X_train, y_train, epochs=20, validation_data=(X_valid, y_valid), verbose=0
)

`fit()` returns a `History` object whose `history` attribute holds the loss and metrics after every epoch. Plotting the training and validation curves together is the standard way to check a training run.

In [ ]:
history_df = pd.DataFrame(history.history)
history_df[["rmse", "val_rmse"]].plot(xlabel="Epoch", ylabel="RMSE", grid=True)
plt.title("Learning curves of the feed-forward network")
plt.show()

history_df.tail(3).round(3)

After 20 epochs the training RMSE is 0.516 and the validation RMSE is 0.526. The two curves stay close together, so the model is not overfitting. The validation curve is noisier than the training curve: near the end it rises to 0.69 for a single epoch and then returns to its previous level. Isolated peaks like this are common with gradient descent, which is why you judge the trend over several epochs rather than one value.

### Evaluate and Predict

In [ ]:
test_mse, test_rmse["Feed-forward network"] = model.evaluate(X_test, y_test, verbose=0)
print(f"Feed-forward network test RMSE: {test_rmse['Feed-forward network']:.3f}")

predictions = model.predict(X_test[:3], verbose=0).ravel()
pd.DataFrame({"Predicted": predictions.round(2), "Actual": y_test[:3].to_numpy()})

## 4. Compare with the Baseline

In [ ]:
pd.Series(test_rmse, name="Test RMSE (100,000 USD)").round(3).to_frame()

The feed-forward network reaches a test RMSE of 0.529, compared with 0.729 for linear regression. In house-price terms, its typical error falls from about 73,000 USD to about 53,000 USD, a reduction of about 27 %. The hidden layers let the model capture non-linear effects, such as house values that depend on location in ways a straight line cannot describe.

The three example predictions in section 3 show that the error varies between districts: the first and third are within 0.21 of the actual value, while the second is off by 0.80, or 80,000 USD.

## Try It: Change the Architecture

Build a new Keras model with a different number of hidden layers, neurons or epochs, train it and compare its **validation** RMSE with the main model's. The validation RMSE changes noticeably from one epoch to the next, so look at the trend of the last few epochs, not only at the final value. The test set is kept for the final comparison, so the Try It does not use it. The normalisation layer is already fitted, so you can reuse it.

In [ ]:
valid_rmse = model.evaluate(X_valid, y_valid, verbose=0)[1]  # the main model on the validation set

keras.utils.clear_session()  # start from a clean Keras state
keras.utils.set_random_seed(42)  # the same starting weights on every run

# The same three layers as the main model, trained twice as long
hidden_layers = [
    keras.layers.Dense(units=50, activation="relu"),
    keras.layers.Dense(units=50, activation="relu"),
    keras.layers.Dense(units=50, activation="relu"),
]
n_epochs = 40

model_try = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        norm_layer,
        *hidden_layers,
        keras.layers.Dense(units=1),
    ]
)
model_try.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)
history_try = model_try.fit(
    X_train, y_train,
    epochs=n_epochs,
    validation_data=(X_valid, y_valid),
    verbose=0,
)

In [ ]:
valid_rmse_try = model_try.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Validation RMSE: {valid_rmse_try:.3f} (main model: {valid_rmse:.3f})")

## Summary

- Stacking dense layers of neurons with a non-linear activation such as ReLU gives a feed-forward network, which can learn relationships a single neuron cannot.
- A Keras `Sequential` model splits the work into building the layers, `compile()` and `fit()`. A `Normalization` layer standardises the features inside the model, so the scaling is saved with it.
- Learning curves for the training and validation sets show whether the model is still improving or starting to overfit.
- On the California housing data the feed-forward network outperforms the linear regression baseline.

Next, continue with [04 - Multiclass Classification with a Feed-Forward Network](04_classification_with_neural_networks.ipynb).

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 10.
- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 4: Classification and regression. [Free to read online](https://deeplearningwithpython.io/).
- Keras guide: [The Sequential model](https://keras.io/guides/sequential_model/).